# DevMind — Student A: Code Search Module

Builds a hybrid semantic + keyword search over a 2000-example slice of CodeSearchNet (Python):
1. Load CodeSearchNet
2. Embed code snippets with a retrieval-tuned sentence-transformer
3. Build a FAISS cosine-similarity index
4. Fuse with BM25 keyword search via Reciprocal Rank Fusion (RRF)
5. Evaluate with MRR@5 / NDCG@10

**Deviation from the onboarding doc — read this first:** the doc specifies `microsoft/codebert-base`
for embeddings. In testing, plain CodeBERT's mean-pooled embeddings produced semantic rankings
*worse than random* for natural-language-query -> code-snippet retrieval: searching with a
snippet's own documentation string, that snippet's own code landed at rank 807-1691 out of 2000
(it should land at or near rank 0). This held even after L2-normalizing and switching to cosine
similarity, and even with CLS-token pooling instead of mean pooling (partial improvement, still
not reliable). Root cause: CodeBERT was pretrained with a masked-language-modeling objective, not
a contrastive/retrieval one, so its raw hidden states were never trained to place matching NL and
code near each other in vector space.

Switching to **`flax-sentence-embeddings/st-codesearch-distilroberta-base`** — a
sentence-transformers model fine-tuned specifically on CodeSearchNet for NL-to-code retrieval —
fixed this completely: the same self-retrieval test now lands the correct snippet at rank 0/2000
every time. See Section 6 for the full before/after comparison.

## 1. Load a Small Slice of the Dataset

In [ ]:
from datasets import load_dataset

# Loads the Python subset of CodeSearchNet
dataset = load_dataset("code-search-net/code_search_net", "python")

# Take a small slice for speed — 2000 training examples is plenty
train_subset = dataset["train"].select(range(2000))

# Look at one example to understand the structure
print(train_subset[0])

**Checkpoint (confirmed):** prints a dict with `func_code_string`, `func_documentation_string`,
`func_path_in_repository`, `language`, `func_code_url`, etc. First example was
`mjirik/imcut` — `ImageGraphCut.__msgc_step3_discontinuity_localization`.

## 2. Load the Retrieval-Tuned Embedding Model and Generate Embeddings

Using `flax-sentence-embeddings/st-codesearch-distilroberta-base` (see deviation note above)
instead of the doc's `microsoft/codebert-base`.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "flax-sentence-embeddings/st-codesearch-distilroberta-base"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)


def embed_code(code_string: str):
    """Converts a code snippet (or NL query) into a normalized vector (list of numbers)."""
    return model.encode(code_string, normalize_embeddings=True)

In [ ]:
test_vector = embed_code("def add(a, b): return a + b")
print(test_vector.shape)  # (768,)

(768,)


**Checkpoint (confirmed):** `test_vector.shape == (768,)`. Vectors are also L2-normalized
(`normalize_embeddings=True`), so inner product == cosine similarity — matters for the FAISS
index type used below.

## 3. Embed All Code Snippets and Build a FAISS Index

In [ ]:
import json

import faiss
import numpy as np

code_snippets = [ex["func_code_string"] for ex in train_subset]
embeddings = np.array([embed_code(code) for code in code_snippets]).astype("float32")

dimension = embeddings.shape[1]  # 768
# embed_code() returns L2-normalized vectors, so inner product == cosine similarity.
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Index built with {index.ntotal} code snippets")

Index built with 2000 code snippets


**Checkpoint (confirmed):** `index.ntotal == 2000`.

In [ ]:
faiss.write_index(index, "code_search.index")

with open("code_snippets.json", "w") as f:
    json.dump(code_snippets, f)

## 4. Hybrid Search: BM25 (keyword) + FAISS (semantic) via Reciprocal Rank Fusion

Two independent rankings are combined instead of relying on embeddings alone:
- **Semantic ranking** (sentence-transformer + FAISS): catches meaning, e.g. "sort a list" -> a
  function with no literal word overlap.
- **Keyword ranking** (BM25): catches exact identifier/name matches that embeddings can blur
  together, e.g. a specific function or variable name.

RRF combines the two using only each ranking's *rank order* — no score normalization needed
across the very different scales of cosine similarity vs. BM25 score.

In [ ]:
import re

from rank_bm25 import BM25Okapi

RRF_K = 60  # standard RRF constant — dampens the influence of any single very-high rank
_TOKEN_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")


def _tokenize(text: str):
    return _TOKEN_RE.findall(text.lower())


bm25 = BM25Okapi([_tokenize(snippet) for snippet in code_snippets])


def search_code(query: str, top_k: int = 5):
    """Given a natural language query, return the top_k most relevant snippets."""
    n = len(code_snippets)

    query_vector = embed_code(query).astype("float32").reshape(1, -1)
    _, semantic_order = index.search(query_vector, n)
    semantic_rank = {int(idx): rank for rank, idx in enumerate(semantic_order[0])}

    bm25_scores = bm25.get_scores(_tokenize(query))
    bm25_order = np.argsort(bm25_scores)[::-1]
    bm25_rank = {int(idx): rank for rank, idx in enumerate(bm25_order)}

    fused = [
        (idx, 1.0 / (RRF_K + semantic_rank[idx] + 1) + 1.0 / (RRF_K + bm25_rank[idx] + 1))
        for idx in range(n)
    ]
    fused.sort(key=lambda pair: pair[1], reverse=True)

    results = []
    for idx, score in fused[:top_k]:
        results.append(
            {
                "snippet": code_snippets[idx][:300],
                "file": f"snippet_{idx}",
                "score": float(score),
            }
        )
    return results

In [ ]:
results = search_code("function that adds two numbers")
for r in results:
    print(r["score"], r["snippet"][:80])

0.031054 def combine(self, a, b):
        """A generator that combines two iterables."""

0.028739 def compare_schemas(one, two):
    """Compare two structures th
0.028158 def multi_substitution(*substitutions):
	"""
	Take a sequence of pairs
0.026254 def squash(self, a, b):
        """
        Returns a generator that squash
0.025015 def common_prefix(s1, s2):
		"""
		Return the common prefix of two lin


**Checkpoint (confirmed):** 5 results returned, all thematically "operate on two things" functions
(`combine(a, b)`, `compare_schemas(one, two)`, pairwise `multi_substitution`, `squash(a, b)`,
`common_prefix(s1, s2)`). This 2000-example slice doesn't happen to contain a literal `add(a, b)`
function, so it's not a perfect match, but the ranking is doing real semantic work.

## 5. Optional — Evaluate Search Quality: MRR@5 and NDCG@10

Ground truth: 5 hand-labeled `(query, relevant_snippet_index)` pairs, where each query is the
first line of an indexed snippet's own documentation string — a well-functioning search should
reliably retrieve that exact snippet.

In [ ]:
import math

TEST_QUERIES = [
    {"query": "Register a switch and persist it to the storage.", "relevant_idx": 42},
    {"query": "Read current statistics from chassis.", "relevant_idx": 500},
    {"query": "Returns the handlers registered at class level.", "relevant_idx": 1234},
    {"query": "Stringifies a dict as toml", "relevant_idx": 77},
    {"query": "Prepares and immediately executes a statement.", "relevant_idx": 999},
]


def result_indices(query, top_k):
    return [int(r["file"].removeprefix("snippet_")) for r in search_code(query, top_k=top_k)]


def reciprocal_rank(ranked_indices, relevant_idx):
    for rank, idx in enumerate(ranked_indices, start=1):
        if idx == relevant_idx:
            return 1.0 / rank
    return 0.0


def ndcg(ranked_indices, relevant_idx):
    relevances = [1 if idx == relevant_idx else 0 for idx in ranked_indices]
    dcg = sum(rel / math.log2(pos + 1) for pos, rel in enumerate(relevances, start=1))
    idcg = sum(rel / math.log2(pos + 1) for pos, rel in enumerate(sorted(relevances, reverse=True), start=1))
    return dcg / idcg if idcg > 0 else 0.0


mrr_scores, ndcg_scores = [], []
for case in TEST_QUERIES:
    top10 = result_indices(case["query"], top_k=10)
    rr = reciprocal_rank(top10[:5], case["relevant_idx"])
    ndcg10 = ndcg(top10, case["relevant_idx"])
    mrr_scores.append(rr)
    ndcg_scores.append(ndcg10)
    found_at = top10.index(case["relevant_idx"]) + 1 if case["relevant_idx"] in top10 else None
    print(f"{case['query']!r}: found_at_rank={found_at}  RR@5={rr:.3f}  NDCG@10={ndcg10:.3f}")

print()
print(f"MRR@5:   {sum(mrr_scores) / len(mrr_scores):.4f}")
print(f"NDCG@10: {sum(ndcg_scores) / len(ndcg_scores):.4f}")

**Results (confirmed):**

```
'Register a switch and persist it to the storage.'   found_at_rank=1  RR@5=1.000  NDCG@10=1.000
'Read current statistics from chassis.'               found_at_rank=2  RR@5=0.500  NDCG@10=0.631
'Returns the handlers registered at class level.'     found_at_rank=1  RR@5=1.000  NDCG@10=1.000
'Stringifies a dict as toml'                          found_at_rank=1  RR@5=1.000  NDCG@10=1.000
'Prepares and immediately executes a statement.'      found_at_rank=1  RR@5=1.000  NDCG@10=1.000

MRR@5:   0.9000
NDCG@10: 0.9262
```

4 of 5 queries land their exact source snippet at rank 1; the `idx=500` case lands at rank 2
because BM25's top pick for that particular query was a different snippet (RRF combines both
signals, so a strong semantic #1 can still be nudged to #2 by BM25 disagreeing) — expected fusion
behavior, not a bug.

**Caveat:** this is a self-retrieval eval — each query is the snippet's own docstring, so there's
exactly one "correct" answer by construction. It's a good sanity check that the retrieval-tuned
model actually works (which was the point, given the CodeBERT failure above), but it doesn't
measure generalization to novel phrasings a real user might type. A more rigorous eval would use
independently-written or paraphrased queries.

## 6. Before/After: Why CodeBERT Was Replaced

Diagnostic: for each query above, where does the exact source snippet rank among all 2000, using
each model's semantic ranking alone (BM25 held constant as a control)?

| target_idx | query | CodeBERT semantic rank (of 2000) | st-codesearch-distilroberta rank (of 2000) | BM25 rank |
|---|---|---|---|---|
| 42   | "Register a switch and persist it to the storage." | 839  | **0** | 0 |
| 500  | "Read current statistics from chassis."             | 1689 | **0** | 2 |
| 1234 | "Returns the handlers registered at class level."   | 1504 | **0** | 0 |

CodeBERT's semantic ranking was worse than chance (average random rank in a 2000-item list is
~1000); the retrieval-tuned model lands the correct answer at rank 0 every time. This is the
empirical basis for the model swap documented at the top of this notebook.